# Load SAE

In [1]:
# set GPU to 2:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [5]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger
import wandb

# If your `NumpyDataModule` is in src/enformer_dataloader_np.py
sys.path.append("src")
# import enformer_dataloader_np
# from enformer_dataloader_np import NumpyDataModule

# Data loader

In [3]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning import LightningDataModule

class NumpyFilesDataset(Dataset):
    def __init__(self, files_dir_or_pattern, limit_batches=None):
        """
        Dataset that loads data from multiple NumPy files.
        
        Args:
            files_dir_or_pattern: Directory containing NumPy files or a glob pattern
            limit_batches: Limit the number of samples for testing/debugging
        """
        # Get list of all NumPy files
        if os.path.isdir(files_dir_or_pattern):
            self.file_paths = sorted(glob.glob(os.path.join(files_dir_or_pattern, "*.np*")))
        else:
            self.file_paths = sorted(glob.glob(files_dir_or_pattern))
            
        print(f"Found {len(self.file_paths)} NumPy files")
        
        # Calculate total samples by examining the first file
        if len(self.file_paths) > 0:
            sample_data = np.load(self.file_paths[0], allow_pickle=True)
            self.samples_per_file = len(sample_data['sequence'])
        else:
            self.samples_per_file = 0
            
        self.total_samples = len(self.file_paths) * self.samples_per_file
        
        # Limit samples if necessary
        if limit_batches is not None:
            self.total_samples = min(self.total_samples, limit_batches)
            
        # Create an index mapping for fast lookup
        self.index_map = {}
        for i in range(min(self.total_samples, len(self.file_paths) * self.samples_per_file)):
            file_idx = i // self.samples_per_file
            sample_idx = i % self.samples_per_file
            self.index_map[i] = (file_idx, sample_idx)
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        file_idx, sample_idx = self.index_map[idx]
        file_path = self.file_paths[file_idx]
        
        # Load data from NumPy file
        data = np.load(file_path, allow_pickle=True)
        
        # Extract sequences and keys
        sequence = torch.from_numpy(data['sequence'][sample_idx]).float()
        key = torch.from_numpy(data['target'][sample_idx]).float()
        
        return (sequence, key)


class NumpyDataModule(LightningDataModule):
    def __init__(self, train_files_pattern, val_files_pattern, test_files_pattern, 
                 batch_size=2, smoke_test=False):
        """
        DataModule for loading data from multiple NumPy files.
        
        Args:
            train_files_pattern: Pattern or directory for training NumPy files
            val_files_pattern: Pattern or directory for validation NumPy files
            test_files_pattern: Pattern or directory for test NumPy files
            batch_size: Batch size for dataloaders
            smoke_test: If True, limit to 10 batches for quick testing
        """
        super().__init__()
        self.train_files_pattern = train_files_pattern
        self.val_files_pattern = val_files_pattern
        self.test_files_pattern = test_files_pattern
        self.batch_size = batch_size
        self.smoke_test = smoke_test

    def setup(self, stage=None):
        batch_limit = 10 if self.smoke_test else None  # Limit to 10 batches for smoke test
        print(f"The batch limit is: {batch_limit}")
        
        self.train_dataset = NumpyFilesDataset(self.train_files_pattern, limit_batches=batch_limit)
        self.val_dataset = NumpyFilesDataset(self.val_files_pattern, limit_batches=batch_limit)
        self.test_dataset = NumpyFilesDataset(self.test_files_pattern, limit_batches=batch_limit)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=4)

# Training

In [ ]:
12288 // 2

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger
import wandb

# -------------------------------------------------------------------------
# 1) Enformer Wrapper for Embeddings
# -------------------------------------------------------------------------
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

class EnformerWithEmbeddings(pl.LightningModule):
    """
    Wraps the Enformer model and captures a specific layer's output.
    """
    def __init__(self, num_tracks=5313, target_layer='transformer.layers.5'):
        super().__init__()
        # Pretrained Enformer
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        
        # HeadAdapter to keep the shape consistent with existing code
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

        self.target_layer = target_layer
        self.hook_store = {}
        self._set_hooks()

    def _set_hooks(self):
        """Register a forward hook on the user-specified target layer."""
        def get_activation(name):
            def hook(module, inp, out):
                # Detach to avoid storing gradients
                self.hook_store[name] = out.detach()
            return hook
        
        # Navigate to the target layer
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        # Register forward hook
        target.register_forward_hook(get_activation(self.target_layer))

    @torch.no_grad()
    def get_embeddings(self, x):
        """
        Forward pass through Enformer to populate hook_store with target layer embeddings.
        Returns (embeddings, predictions).
        """
        _ = self.model(x)  # triggers forward hooks
        emb = self.hook_store[self.target_layer]
        return emb, _

    def forward(self, x):
        return self.model(x)
    
# -------------------------------------------------------------------------
# 2) Sparse Autoencoder
# -------------------------------------------------------------------------
class SparseAutoencoder(nn.Module):
    """
    A basic MLP-based autoencoder with an L1 penalty on the hidden activations
    to encourage sparsity.
    """
    def __init__(self, input_dim, hidden_dim=12288, l1_weight=1e-5):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.l1_weight = l1_weight

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            # No final activation unless needed
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

# -------------------------------------------------------------------------
# 3) Combined LightningModule to Train AE on-the-fly w/ Enformer
# -------------------------------------------------------------------------
class OnTheFlyAETrainer(pl.LightningModule):
    """
    This module:
      - Uses EnformerWithEmbeddings to get embeddings on-the-fly.
      - Passes them to SparseAutoencoder.
      - Computes MSE + L1 sparse penalty on the hidden layer.
    """
    def __init__(
        self,
        enformer_target_layer="conv_tower.5.2.to_attn_logits",
        input_dim=1536,
        hidden_dim=12288,
        l1_weight=1e-5,
        lr=1e-3
    ):
        super().__init__()
        self.save_hyperparameters()

        # Enformer for embeddings (frozen)
        self.enformer_model = EnformerWithEmbeddings(
            target_layer=enformer_target_layer
        )

        # Sparse Autoencoder
        self.ae = SparseAutoencoder(
            input_dim=input_dim, 
            hidden_dim=hidden_dim, 
            l1_weight=l1_weight
        )

        self.lr = lr

    def forward(self, x):
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(x)
        # Permute from [B, H, W, C] to [B, C, H, W]
        # emb = emb.permute(0, 3, 1, 2)
        # Adaptive pooling to target spatial size (24,16) so that 2*24*16=768
        # emb = F.adaptive_avg_pool2d(emb, (24, 16)) # summarize accross 5 bins
        # emb = emb.flatten(start_dim=1)
        emb = emb[:, :, emb.shape[2] // 2]
        print(emb.shape)
        x_hat, z = self.ae(emb)
        return x_hat, z

    def training_step(self, batch, batch_idx):
        sequences, _ = batch
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(sequences)
        # Permute to channels-first: [B, H, W, C] -> [B, C, H, W]
        # emb = emb.permute(0, 3, 1, 2)
        # emb = F.adaptive_avg_pool2d(emb, (24, 16))
        # emb = emb.flatten(start_dim=1)

        emb = emb[:, :, emb.shape[2] // 2]
        print(emb.shape)

        x_hat, z = self.ae(emb)
        recon_loss = F.mse_loss(x_hat, emb)
        l1_loss = z.abs().mean()
        loss = recon_loss + self.ae.l1_weight * l1_loss

        self.log("train_recon_loss", recon_loss, prog_bar=True)
        self.log("train_l1_loss", l1_loss, prog_bar=True)
        self.log("train_loss", loss, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        sequences, _ = batch
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(sequences)
        # emb = emb.permute(0, 3, 1, 2)
        # emb = F.adaptive_avg_pool2d(emb, (24, 16))
        # emb = emb.flatten(start_dim=1)

        emb = emb[:, :, emb.shape[2] // 2]

        x_hat, z = self.ae(emb)
        recon_loss = F.mse_loss(x_hat, emb)
        l1_loss = z.abs().mean()
        loss = recon_loss + self.ae.l1_weight * l1_loss

        self.log("val_recon_loss", recon_loss, prog_bar=True)
        self.log("val_l1_loss", l1_loss, prog_bar=True)
        self.log("val_loss", loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# -------------------------------------------------------------------------
# 4) Example Usage in a Jupyter Notebook
# -------------------------------------------------------------------------
def run_training_jupyter():
    """
    Example function showing how to set up everything in a single notebook cell.
    Adjust paths and hyperparameters to your needs.
    """
    wandb_logger = WandbLogger(project="enformer-sparse-autoencoder", log_model=True)

    # from enformer_dataloader_np import NumpyDataModule
    data_module = NumpyDataModule(
        train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train",
        val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid",
        test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test",
        batch_size=4,
    )
    data_module.setup()

    model = OnTheFlyAETrainer(
        enformer_target_layer="conv_tower.5.2.to_attn_logits",
        input_dim=1536,      # 2 channels * 24 * 16 = 768
        hidden_dim=12_288,
        l1_weight=1e-4,
        lr=1e-3
    )

    checkpoint_cb = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=3,
        dirpath="checkpoints/",
        filename="sae-{epoch:02d}-{val_loss:.4f}"
    )
    early_stop_cb = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min"
    )

    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        logger=wandb_logger,
        callbacks=[checkpoint_cb, early_stop_cb],
        deterministic=False,
        log_every_n_steps=5
    )

    trainer.fit(model, data_module)
    trainer.test(model, data_module)
    wandb.finish()

    return trainer, model

# Run training in the notebook
run_training_jupyter()


The batch limit is: None
Found 1356 NumPy files
Found 291 NumPy files
Found 290 NumPy files


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/rajesh/projects/hackathon/SAE_Hackathon/.venv/lib64/python3.9/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/home/rajesh/projects/hackathon/SAE_Hackathon/.venv/lib64/python3.9/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/rajesh/projects/hackathon/SAE_Hackathon/notebooks/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [2]

  | Name           | Type                   | Params | Mode 
------------------------------------------------------------------
0 | enformer_model | EnformerWithEmbeddings | 267 M  | train
1 | ae             | SparseAutoencoder      | 37.8 M | train
-------------------------------

The batch limit is: None
Found 1356 NumPy files
Found 291 NumPy files
Found 290 NumPy files


RuntimeError: mat1 and mat2 shapes cannot be multiplied (6144x2 and 1536x12288)

In [7]:
6144*2

12288

In [10]:
768*256

18874368

In [ ]:
d = enformer_model.state_dict()

for key in d:
    print(key)

In [ ]:
# enformer_target_layer = "conv_tower.5.2.to_attn_logits"
# enformer_target_layer = "transformer.10.1.fn.4"
enformer_target_layer = "conv_tower.5.1.fn.2"
enformer_model = EnformerWithEmbeddings(
        target_layer=enformer_target_layer
    )
x = torch.randint(0, 5, (1, 196_608)) # for ACGTN, in that order (-1 for padding)

with torch.no_grad():
    emb, _ = enformer_model.get_embeddings(x)

emb.shape

In [ ]:
emb.shape[2] / 2

In [ ]:
emb[:, :, 1536].shape

# Evaluate Model

# Interpret model